## FUNCTIONAL PROGRAMMING - FINAL PROJECT 26-27
José Manuel Cosano (100523050) & Mateo Nonzioli (100523033)

1. Type definitions for all concepts in the statement

In [ ]:
-- Descriptors
type Description = String
data Type = Private | Doctors | Work deriving (Show, Eq)

-- Date definition
-- Dedicated data type for the month 
data Month = January | February | March | April
    | May | June | July | August
    | September | October | November | December deriving (Eq, Enum, Ord)

instance Show Month where
    show January   = "January"
    show February  = "February"
    show March     = "March"
    show April     = "April"
    show May       = "May"
    show June      = "June"
    show July      = "July"
    show August    = "August"
    show September = "September"
    show October   = "October"
    show November  = "November"
    show December  = "December"

data IsWorkDay = WorkingDay | Holiday deriving (Show, Eq)

-- Date follows the format (Day Month Year IsWorkDay) for valid dates with Day and Year integers and Month as above
data Date = ValidDate Int Month Int IsWorkDay | InvalidDate deriving Eq

-- Override Show instance for Date (accounting for invalid dates)
instance Show Date where
    show InvalidDate = "Invalid date"
    show (ValidDate d m y isworkday) =
        -- Here "show m" uses the Show Month instance defined above
        show d ++ " " ++ show m ++ " " ++ show y ++ " (" ++ show isworkday ++ ")" 



-- Time definition. We use the Maybe type to allow for optional arguments
type HoursMinutesTime = (Int, Int)
data Time = Time {start :: Maybe HoursMinutesTime, end :: Maybe HoursMinutesTime, duration :: Maybe Int} deriving Eq

-- Override Show instance for Time
instance Show Time where
    show (Time s e d) =
        "Start: " ++ showTimeValue s ++
        ", End: " ++ showTimeValue e ++
        ", Duration: " ++ showDuration d


-- For start and end, show already handles Maybe, since HoursMinutesTime is made of two Int (which have Show instances)
-- Function handles not specified time and removes Just
showTimeValue :: Maybe HoursMinutesTime -> String
showTimeValue Nothing = "Not specified"
showTimeValue (Just (h, m)) =
    show h ++ ":" ++ (if m < 10 then "0" else "") ++ show m

-- Function handles not specified duration and removes Just
showDuration :: Maybe Int -> String
showDuration Nothing = "Not specified"
showDuration (Just d) = show d ++ " minutes"

-- List of participants
newtype Person = Person String deriving (Show, Eq)
type Participants = [Person]

-- Frequency of the event for REPEATED events ONLY (daily, weekly, working days)
data Repetition = Daily | Weekly | WorkDays deriving (Show, Eq)

-- Distinguish between Punctual and Repeated events
data Repetitions = Punctual | Repeated Repetition Date deriving (Eq)

-- Override Show instance for Repetitions
instance Show Repetitions where
    show Punctual = "No repetition"
    show (Repeated Daily d) =
        "Daily"
    show (Repeated Weekly d) =
        "Weekly"
    show (Repeated WorkDays d) =
        "Every working day"

-- Complete constructor for events
data Event = Event
    { description :: Description
    , eventType :: Type
    , date :: Maybe Date -- Date is specified as optional
    , time :: Time
    , participants :: Participants
    , repetitions :: Repetitions
    } deriving Eq

2. Function to display events in a pretty way

In [ ]:
-- Override Show class for Event
instance Show Event where
    show (Event desc typ dat tim parts reps) =
        "Description: " ++ desc ++ "\n" ++
        "Type: " ++ show typ ++ "\n" ++
        "Date: " ++ showDate dat ++ "\n" ++
        "Time: " ++ show tim ++ "\n" ++
        "Participants: " ++ showParticipants parts ++ "\n" ++
        "Repetitions: " ++ show reps


-- Function handles not specified date and removes Just
showDate :: Maybe Date -> String
showDate Nothing = "Not specified"
showDate (Just d) = show d

-- Function removes brackets from Show of lists
showParticipants :: Participants -> String
showParticipants [] = ""
showParticipants [Person name] = name
showParticipants (Person name : xs) =
    name ++ ", " ++ showParticipants xs



3. isWorkingDate function

In [ ]:
-- Days of the week
data WeekDay = Monday | Tuesday | Wednesday | Thursday | Friday | Saturday | Sunday deriving (Show, Eq, Enum)

-- Number of days of each month (leap years are ignored)
daysInMonth :: Month -> Int
daysInMonth January   = 31
daysInMonth February  = 28
daysInMonth March     = 31
daysInMonth April     = 30
daysInMonth May       = 31
daysInMonth June      = 30
daysInMonth July      = 31
daysInMonth August    = 31
daysInMonth September = 30
daysInMonth October   = 31
daysInMonth November  = 30
daysInMonth December  = 31

-- Position of a day within its year (1 January = 1): the days of all the previous months plus the day of the month
dayOfYear :: Int -> Month -> Int
dayOfYear d m = sum (map daysInMonth (filter (< m) [January ..])) + d

-- Weekday of a given day of the year, knowing the weekday of 1 January.
-- Moving n-1 days forward from 1 January and wrapping around every 7 days.
weekDayOf :: WeekDay -> Int -> WeekDay
weekDayOf jan1 n = toEnum ((fromEnum jan1 + n - 1) `mod` 7)

-- Saturdays and Sundays are not working days
isWeekend :: WeekDay -> Bool
isWeekend Saturday = True
isWeekend Sunday   = True
isWeekend _        = False

-- Two dates are the same day if day, month and year match
sameDay :: Date -> Date -> Bool
sameDay (ValidDate d1 m1 y1 _) (ValidDate d2 m2 y2 _) = d1 == d2 && m1 == m2 && y1 == y2
sameDay _ _ = False

-- A date is a working day if it is a valid date that is neither a weekend
-- nor in the list of holidays. The WeekDay is the one of 1 January of the
-- year of the date.
isWorkingDate :: Date -> [Date] -> WeekDay -> Bool
isWorkingDate InvalidDate _ _ = False
isWorkingDate dt@(ValidDate d m _ _) holidays jan1 = not (isWeekend (weekDayOf jan1 (dayOfYear d m))) && not (any (sameDay dt) holidays)

4. completeTime function that completes the timing of an Event: if it lacks a duration, it calculates and
inserts it from the start and end times; if it has a start time and duration, it calculates the end time, and
vice versa. If it only specifies a duration, it leaves the Event unchanged as it is not yet scheduled.

In [ ]:
-- Auxiliary functions to convert time to/from minutes
toMinutes :: HoursMinutesTime -> Int
toMinutes (h, m) = 60 * h + m 

fromMinutes :: Int -> Maybe HoursMinutesTime
fromMinutes timeMin 
    | timeMin < 0 || timeMin >= 24*60 = Nothing
    | otherwise = Just (timeMin `div` 60, timeMin `mod` 60) 
    -- We use `div` (integer division) to display the hour as an Int, with the minutes displayed separately

-- Separate function to complete the time whenever needed
completeEvTime :: Time -> Time

-- start and end times provided => convert to minutes and subtract
completeEvTime t@(Time (Just start) (Just end) Nothing)
    | computedDuration >= 0 = t {duration = Just computedDuration}
    | otherwise = t
    where
        computedDuration = toMinutes end - toMinutes start

-- start and duration provided
completeEvTime t@(Time (Just start) Nothing (Just duration))
    | duration >= 0 = t {end = fromMinutes (toMinutes start + duration)}
    | otherwise = t

-- end and duration provided
completeEvTime t@(Time Nothing (Just end) (Just duration))
    | duration >= 0 = t {start = fromMinutes (toMinutes end - duration)}
    | otherwise = t

-- in any other case, return the same time
completeEvTime t = t

completeTime :: Event -> Event
completeTime ev = ev {time = completeEvTime(time ev)}

5. A listAgenda Agenda function that returns all the Events of an Agenda sorted from earliest to latest

In [ ]:
-- An agenda: the owner and the list of events
data Agenda = Agenda { owner :: Person, events :: [Event] } deriving (Show, Eq)

-- (year, month, day, hours, minutes)
type SortKey = (Int, Int, Int, Int, Int)

-- Chronological key of an event, or Nothing if it is not scheduled
eventKey :: Event -> Maybe SortKey
eventKey ev = makeKey (date ev) (start (time (completeTime ev)))

-- Builds the key from the date and the start time of an event
makeKey :: Maybe Date -> Maybe HoursMinutesTime -> Maybe SortKey
makeKey (Just (ValidDate d m y _)) (Just (h, mi)) = Just (y, fromEnum m, d, h, mi)
makeKey _ _ = Nothing

-- Order of the listing: scheduled events first, chronologically; then the unscheduled ones, alphabetically by description (case-sensitive).
comesBefore :: Event -> Event -> Bool
comesBefore e1 e2 = keyBefore (eventKey e1) (eventKey e2)
    where
        keyBefore :: Maybe SortKey -> Maybe SortKey -> Bool
        keyBefore (Just k1) (Just k2) = k1 <= k2
        keyBefore (Just _)  Nothing   = True
        keyBefore Nothing   (Just _)  = False
        keyBefore Nothing   Nothing   = description e1 <= description e2

-- Inserts an event into an already sorted list, keeping it sorted
insertSorted :: Event -> [Event] -> [Event]
insertSorted ev [] = [ev]
insertSorted ev (x:xs)
    | comesBefore ev x = ev : x : xs
    | otherwise        = x : insertSorted ev xs

-- Insertion sort of a list of events
sortEvents :: [Event] -> [Event]
sortEvents = foldr insertSorted []

-- All the events of an agenda, from earliest to latest
listAgenda :: Agenda -> [Event]
listAgenda ag = sortEvents (events ag)


6. searchEvents function that returns all the events of the Agenda with such list of descriptors

In [ ]:
-- Search criteria for filtering events
data Descriptor = ByDesc Description | ByType Type | ByDate Date | ByStartTime HoursMinutesTime | ByEndTime HoursMinutesTime
    | ByDuration Int | ByReps Repetitions | ByPerson Person | ByParticipants Participants -- event includes all listed participants
    deriving (Show, Eq)

-- Check whether a single Event matches a single Descriptor
matchesDescriptor :: Event -> Descriptor -> Bool
matchesDescriptor ev desc
    | desc == ByDesc d             = description ev == d
    | desc == ByType t             = eventType ev == t
    | desc == ByDate dt            = date ev == Just dt
    | desc == ByStartTime st       = start (time completedEv) == Just st
    | desc == ByEndTime e          = end (time completedEv) == Just e
    | desc == ByDuration dur       = duration (time completedEv) == Just dur
    | desc == ByReps rep           = repetitions ev == rep
    | desc == ByPerson p           = p `elem` participants ev
    | desc == ByParticipants parts = all (`elem` participants ev) parts
  where
    completedEv = completeTime ev

-- Return all events in the Agenda that satisfy every Descriptor in the list
-- We use 'filter' (a higher order function) and 'all' to select events that match all descriptors
searchEvents :: [Descriptor] -> Agenda -> [Event]
searchEvents descriptors agenda =
    filter (\ev -> all (matchesDescriptor ev) descriptors) (events agenda)

7. deleteEvents function that returns an Agenda where the events having those descriptors are removed

In [ ]:
-- With Descriptor as above
deleteEvents :: [Descriptor] -> Agenda -> Agenda
deleteEvents descriptors agenda =
    agenda { events = filter (\ev -> not (all (matchesDescriptor ev) descriptors)) (events agenda) }
    -- Which is the exact opposite of searchEvents => same logic but negating the condition